## Environmental Performance Model

# T3.3 EIT-001 Environmental Monitoring Tool

READ.ME

This notebook calls the databases and project defined in EIT-003 and handles the import and integration of sensor data into the LCIs.

- Setting the project 
- Relevant sensor data manipulation 
- Integration in foreground database as scenario
- LCA results in DPP format

> The functionalities represent the requirements for EU REED WP3 T3.3 EIT-001 deliverable. Notebook depends on 1) EIT-003 Environmental Performance Model

### 1. Set the project

In [95]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
from bw2io.strategies.generic import set_code_by_activity_hash


import matplotlib.pyplot as plt
import d3graph
import polyviz

import json
from datetime import date

In [96]:
config = Config()

In [97]:
bd.projects.set_current(config.PROJECT_NAME)

In [98]:
bd.databases

Databases dictionary with 4 object(s):
	BONSAI_V2.1.6
	BONSAI_V2.1.6 biosphere
	ecoinvent-3.10-biosphere
	reed

In [99]:
bonsai = bd.Database('BONSAI_V2.1.6')
bonsai_bio = bd.Database('BONSAI_V2.1.6 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')
reed = bd.Database('reed')

### 2. Sensor data integration

#### 2.1 Import sensor data

This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [100]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [101]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [102]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    monitoring_data_path,
    tool_descriptions_path,
)
print(sensordata)

C:\Users\TimWeber\repos_20LCA\REED\reed\src\reed\sensordata_analysis.py:144: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


                                                  energy_consumption  \
company event_id                     Tool_Number                       
goimek  OF10001_G_BQC_S8CF2G         0.0                    8.154403   
                                     1.0                   24.300319   
                                     2.0                   10.289347   
                                     3.0                   14.673875   
                                     4.0                    0.930889   
...                                                              ...   
wwr     OF10005_SETUPS2_G_TWB_7SZ421 2694.0                 2.915730   
                                     2695.0                 1.068689   
                                     2696.0                 2.709187   
                                     2722.0                 0.631658   
                                     3350.0                23.502143   

                                                  Power_Active 

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [103]:
print(ambiguous_company_tools)

   company  Tool_Number                   op_type_candidates
0   goimek            2                 [AUXILIARY, MILLING]
1   goimek            6                 [AUXILIARY, MILLING]
2   goimek           30                 [AUXILIARY, MILLING]
3   goimek           31                   [CHAMFER, MILLING]
4   goimek           44                 [AUXILIARY, MILLING]
5   goimek           45                 [AUXILIARY, MILLING]
6  komatsu            3           [AUXILIARY, ROUGH MILLING]
7  komatsu           48  [AUXILIARY, MILLING, ROUGH MILLING]
8  komatsu           50        [ROUGH BORING, ROUGH MILLING]
9  komatsu           72      [FINISH MILLING, ROUGH MILLING]


In [104]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)

In [105]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)

#### 2.2 Scenario setup for sensor data integation

this section is missing a part about how to handle the processes that are not in hours?

In [106]:
from reed.swapping_data import (
    find_activities_by_name,
    build_scenarios_from_power_active,
    build_consumer_input_scenarios_from_hours,
    describe_scenarios,
    lcia_for_activity_with_scenario,
    find_subactivity_by_name_fragment,
    build_consumer_input_scenarios_from_hours_for_activity,
    get_product_id_for_activity,
)

Select which event you want to analyse

In [107]:
company = "goimek"
event_id = "OF10005_G_BQC_S8CF2G"
product = "GOIMEK total"
demand = 1

Start by selected the company and event_id where you want to swap in the sensordata

In [108]:
#Split sensordata by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[(company, event_id)]
)
print(tool_energy_by_company_and_event)

                                                          energy_consumption  \
company event_id             reed tool                                         
goimek  OF10005_G_BQC_S8CF2G drilling, soraluce                    28.517208   
                             generic machining, soraluce           81.574639   
                             milling, soraluce                     69.676208   

                                                              hours  \
company event_id             reed tool                                
goimek  OF10005_G_BQC_S8CF2G drilling, soraluce           50.165556   
                             generic machining, soraluce   6.523333   
                             milling, soraluce             5.779167   

                                                          Power_Active  
company event_id             reed tool                                  
goimek  OF10005_G_BQC_S8CF2G drilling, soraluce               0.568462  
                        


Find all processes present in the sensordata in the reed-database

In [109]:
tools = tool_energy_by_company_and_event.index.get_level_values("reed tool").unique()
found = {t: find_activities_by_name(reed, t, exact=False) for t in tools}
found

{'drilling, soraluce': ['drilling, soraluce' (hour, ES, None),
  'drilling, soraluce' (hour, GB, None),
  'drilling, soraluce' (hour, DE, None),
  'drilling, soraluce' (kilogram, DE, None),
  'drilling, soraluce' (kilogram, ES, None),
  'drilling, soraluce' (kilogram, GB, None)],
 'generic machining, soraluce': ['generic machining, soraluce' (kilogram, DE, None),
  'generic machining, soraluce' (kilogram, GB, None),
  'generic machining, soraluce' (hour, GB, None),
  'generic machining, soraluce' (hour, ES, None),
  'generic machining, soraluce' (kilogram, ES, None),
  'generic machining, soraluce' (hour, DE, None)],
 'milling, soraluce': ['milling, soraluce' (kilogram, ES, None),
  'milling, soraluce' (kilogram, DE, None),
  'milling, soraluce' (hour, GB, None),
  'milling, soraluce' (hour, DE, None),
  'milling, soraluce' (kilogram, GB, None),
  'milling, soraluce' (hour, ES, None)]}

Find subactivities called A3 for the product under study

In [110]:
product_activity = reed.get(name=product)

# Find the A3 activity inside KOMATSU total
a3_matches = find_subactivity_by_name_fragment(
    parent_activity=product_activity,
    name_fragment="A3",
)

print(a3_matches)
a3_activity = a3_matches[0]

['GOIMEK A3: machining' (unit, ES, None)]


In [111]:
soraluce_matches = find_subactivity_by_name_fragment(
    parent_activity=a3_activity,
    name_fragment="soraluce machining",
)

print(soraluce_matches)

soraluce_activity = soraluce_matches[0]

['Soraluce machining, GOIMEK' (unit, ES, None)]


In [112]:
#Identifying the electricity activity in bonsai to be able to compute the original electricity input amount for the activities found above
electricity = bonsai.get(name='production of electricity (market for)', location='GB', unit='TJ')

Create scenario where alle electricity inputs from the sensordata to the processes identified in found.

In [113]:
scenarios = build_scenarios_from_power_active(tool_energy_by_company_and_event, found, electricity)
scenarios


original  goimek_OF10005_G_BQC_S8CF2G
product            activity                                                 
318672587033227265 318722215241920512  0.000000                     0.000002
                   318722214809907200  0.000015                     0.000045
                   318722215044788224  0.000015                     0.000043

The input of products in found to consuming activities are found and used to build scenarios.

In [114]:
consumer_scenarios = build_consumer_input_scenarios_from_hours_for_activity(
    tool_energy_by_company_and_event,
    found,
    consumer_activity=soraluce_activity,
)
consumer_scenarios

,,original,goimek_OF10005_G_BQC_S8CF2G
product,activity,,
318722215241920512,318722217624285184,50.165556,50.165556
318722214990262272,318722217624285184,6.523333,6.523333
318722215191588864,318722217624285184,5.779167,5.779167


In [115]:
full_scenario = pd.concat([scenarios, consumer_scenarios])

#### 2.3 Scenario data scenario setup

In [116]:
full_scenario_readable = describe_scenarios(full_scenario)
full_scenario_readable

,product_name,product_unit,activity_name,activity_unit,original,goimek_OF10005_G_BQC_S8CF2G
0,production of electricity (market for),TJ,"drilling, soraluce",hour,0.000000,0.000002
1,production of electricity (market for),TJ,"generic machining, soraluce",hour,0.000015,0.000045
2,production of electricity (market for),TJ,"milling, soraluce",hour,0.000015,0.000043
3,"drilling, soraluce",hour,"Soraluce machining, GOIMEK",unit,50.165556,50.165556
4,"generic machining, soraluce",hour,"Soraluce machining, GOIMEK",unit,6.523333,6.523333
5,"milling, soraluce",hour,"Soraluce machining, GOIMEK",unit,5.779167,5.779167


#### 2.4 Initial LCA scores for Monitoring Footprint

LCA score with sensor data

In [117]:
score, lca = lcia_for_activity_with_scenario(
    activity=product_activity,
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    scenario_col="original",     # "original" or None (for defined scenario)
    amount=demand
)
score

16390.086623457777

In [118]:
score, lca = lcia_for_activity_with_scenario(
    activity=product_activity,
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    scenario_col=None,     # "original" or None (for defined scenario)
    amount=demand
)
score

16399.145563060705

LCA score without sensor data

In [119]:
bwa.print_recursive_calculation(product_reed,
lcia_method=config.lcia_gwp100,max_level=1,cutoff=0.001)

Fraction of score | Absolute score | Amount | Activity
0001 | 1.171e+04 |     1 | 'KOMATSU total' (unit, GB, None)
  0.899 | 1.053e+04 |     1 | 'KOMATSU A1: raw materials' (unit, GB, None)
  0.0171 | 200.4 |     1 | 'KOMATSU A2: transport' (unit, GB, None)
  0.113 | 1,325 |     1 | 'KOMATSU A3: machining' (unit, GB, None)
  -0.0297 |  -348 | 0.1824 | 'treatment of steel, industrial waste (market for)' (ton, GB, None)


### 3. Monitoring Stage (Finished=0)

During the production stage, indicated by the column "finished=0", the client wants to track total climate impacts and impacts per machining operation. The input data is expected to be aggregated per operation.

In [120]:
import importlib
import reed.LCA_functions

importlib.reload(reed.LCA_functions)
from reed.LCA_functions import (get_machining_activity, monitoring_footprint_output)

In [121]:
monitoring_footprint_output_unfinished = monitoring_footprint_output(product_activity, lcia_method=config.lcia_gwp100, company=company, event_id=event_id)
print(json.dumps(monitoring_footprint_output_unfinished, indent=2))

Exported to monitoring_goimek_OF10005_G_BQC_S8CF2G_2026-06-01.json
{
  "product": "GOIMEK total",
  "total_climate_impact_kg_co2eq": 16390.0867,
  "machining_contributions": [
    {
      "operation": "generic machining, soraluce",
      "amount": 6.523333
    },
    {
      "operation": "drilling, soraluce",
      "amount": 50.165556
    },
    {
      "operation": "milling, soraluce",
      "amount": 5.779167
    },
    {
      "operation": "soraluce machine, Goimek",
      "amount": 9.1e-05
    }
  ]
}


### 4. Digital Product Passport Export (Finished=1)

#### 4.1 Functions for DPP alignment

In [122]:
def lcia_score_for_named_input(
    parent_activity,
    name_fragment,
    method,
    scenario_df=None,
    scenario_col=None,
    amount=1.0,
):
    """
    Calculate LCIA score for direct inputs of parent_activity whose names contain name_fragment.
    Applies scenario_df if provided.
    """
    matches = [
        exc for exc in parent_activity.technosphere()
        if name_fragment.lower() in exc.input.get("name", "").lower()
    ]

    if not matches:
        raise ValueError(
            f"No input containing '{name_fragment}' found in {parent_activity.get('name')}"
        )

    total_score = 0

    for exc in matches:
        input_activity = exc.input
        input_amount = float(exc["amount"]) * amount

        if scenario_df is None:
            fu, data_objs, _ = bd.prepare_lca_inputs(
                {input_activity: input_amount},
                method=method,
            )

            lca = bc.LCA(fu, method=method, data_objs=data_objs)
            lca.lci()
            lca.lcia()

            total_score += lca.score

        else:
            score, _ = lcia_for_activity_with_scenario(
                activity=input_activity,
                method=method,
                scenario_df=scenario_df,
                scenario_col=scenario_col,
                amount=input_amount,
            )

            total_score += score

    return total_score

In [123]:
score_A1 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A1",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

score_A2 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A2",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

score_A3 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A3",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

In [124]:
score_total = score_A1 + score_A2 + score_A3
score_total

17968.290766901602

#### 4.2 To be shared with TUDa

In [126]:
activity = bd.Database('reed').get(name=product)
location = activity.get("location")
unit = activity.get("unit")

def make_pcf_entry(score, life_cycle_phase):
    return {
        "PcfCalculationMethods": [
            {
                "PcfCalculationMethod": "ISO 14044"
            }
        ],
        "PCF in kg CO2eq": round(score, 1),
        "ProcessName": f"{product} {life_cycle_phase}",
        "ReferenceValueForCalculation": unit,
        "QuantityOfMeasureForCalculation": demand,
        "LifeCyclePhases": [
            {
                "LifeCyclePhase": life_cycle_phase
            }
        ],
        "ExplanatoryStatement": (
            "Product Carbon Footprint associated with the cradle-to-gate life cycle phases (A1-A3) of the product, calculated using the specific production data provided by the company for this product."
        ),
        "GoodsHandoverAddress": {
            "Country": location
        },
        "PublicationDate": date.today().isoformat(),
        # "ExpirationDate": "2027-03-06"
    }

dpp_output = {
    "DPP": [
        {
            "company": company,
            "event_id": event_id,
            "product": product,
            "carbonFootprint": {
                "ProductCarbonFootprints": [
                    make_pcf_entry(score_A1, "A1"),
                    make_pcf_entry(score_A2, "A2"),
                    make_pcf_entry(score_A3, "A3"),
                    make_pcf_entry(score_A1 + score_A2 + score_A3, "A1-A3")  # optional total
                ]
            }
        }
    ]
}

dpp_output


{'DPP': [{'company': 'goimek',
   'event_id': 'OF10005_G_BQC_S8CF2G',
   'product': 'GOIMEK total',
   'carbonFootprint': {'ProductCarbonFootprints': [{'PcfCalculationMethods': [{'PcfCalculationMethod': 'ISO 14044'}],
      'PCF in kg CO2eq': 16819.5,
      'ProcessName': 'GOIMEK total A1',
      'ReferenceValueForCalculation': 'unit',
      'QuantityOfMeasureForCalculation': 1,
      'LifeCyclePhases': [{'LifeCyclePhase': 'A1'}],
      'ExplanatoryStatement': 'Product Carbon Footprint associated with the cradle-to-gate life cycle phases (A1-A3) of the product, calculated using the specific production data provided by the company for this product.',
      'GoodsHandoverAddress': {'Country': 'ES'},
      'PublicationDate': '2026-06-01'},
     {'PcfCalculationMethods': [{'PcfCalculationMethod': 'ISO 14044'}],
      'PCF in kg CO2eq': 227.7,
      'ProcessName': 'GOIMEK total A2',
      'ReferenceValueForCalculation': 'unit',
      'QuantityOfMeasureForCalculation': 1,
      'LifeCyclePha

In [127]:
# Export to JSON file
filename = f"DPP_{company}_{event_id}_{date.today().isoformat()}.json"

with open(filename, "w") as f:
    json.dump(dpp_output, f, indent=2)

print(f"Exported to {filename}")

Exported to DPP_goimek_OF10005_G_BQC_S8CF2G_2026-06-01.json
